In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

In [4]:
project_dir = Path.cwd().parent

if project_dir.name == "notebook":
    project_dir = project_dir.parent

file_path = project_dir / "data" / "raw" / "smartphones.csv"

df = pd.read_csv(file_path)

print("Data Load Successfully")


Data Load Successfully


In [5]:
#check numeric columns for usual values
df.describe().T

,count,mean,std,min,25%,50%,75%,max
price,980.0,32520.504082,39531.812669,3499.00,12999.00,19994.50,35491.50,650000.00
avg_rating,879.0,7.825825,0.740285,6.00,7.40,8.00,8.40,8.90
5G_or_not,980.0,0.560204,0.496616,0.00,0.00,1.00,1.00,1.00
num_cores,974.0,7.772074,0.836845,4.00,8.00,8.00,8.00,8.00
processor_speed,938.0,2.427217,0.464090,1.20,2.05,2.30,2.84,3.22
battery_capacity,969.0,4817.748194,1009.540054,1821.00,4500.00,5000.00,5000.00,22000.00
fast_charging_available,980.0,0.854082,0.353205,0.00,1.00,1.00,1.00,1.00
fast_charging,769.0,46.126138,34.277870,10.00,18.00,33.00,66.00,240.00
ram_capacity,980.0,6.560204,2.744378,1.00,4.00,6.00,8.00,18.00
internal_memory,980.0,141.036735,107.134516,8.00,64.00,128.00,128.00,1024.00


Create a suspicious-values report

In [6]:
import pandas as pd

suspicious_report = []

# check values that violate basic logical expectations

checks = {
    "price" : df["price"] < 0,
    "battery_capacity": df["battery_capacity"] < 0 ,
    "processor_speed" : df["processor_speed"] < 0,
    "avg_rating": ~df["avg_rating"].between(0,10),

}

for column, condition in checks.items():
    flagged_rows = df.loc[condition.fillna(False)]

    for index, row in flagged_rows.iterrows():
        suspicious_report.append({
            "row_index": index,
            "column" : column,
            "value": row[column],
            "reason": "Outside the expected basic range",
            "status": "Needs investigation"
        })

suspicious_df = pd.DataFrame(
    suspicious_report,
    columns=[
        "row_index",
        "column",
        "value",
        "reason",
        "status"
    ]
)

suspicious_df


,row_index,column,value,reason,status
0,40,avg_rating,NaN,Outside the expected basic range,Needs investigation
1,43,avg_rating,NaN,Outside the expected basic range,Needs investigation
2,50,avg_rating,NaN,Outside the expected basic range,Needs investigation
3,55,avg_rating,NaN,Outside the expected basic range,Needs investigation
4,59,avg_rating,NaN,Outside the expected basic range,Needs investigation
...,...,...,...,...,...
96,857,avg_rating,NaN,Outside the expected basic range,Needs investigation
97,873,avg_rating,NaN,Outside the expected basic range,Needs investigation
98,874,avg_rating,NaN,Outside the expected basic range,Needs investigation
99,875,avg_rating,NaN,Outside the expected basic range,Needs investigation


Record missing values as a separate issue

In [7]:
missing_report = (
    df.isna()
      .sum()
      .rename("missing_count")
      .reset_index()
      .rename(columns={"index": "column"})
)

missing_report = missing_report[
    missing_report["missing_count"] > 0
].copy()

missing_report["status"] = "Needs investigation"

missing_report

,column,missing_count,status
3,avg_rating,101,Needs investigation
5,processor_brand,20,Needs investigation
6,num_cores,6,Needs investigation
7,processor_speed,42,Needs investigation
8,battery_capacity,11,Needs investigation
10,fast_charging,211,Needs investigation
16,os,14,Needs investigation
18,primary_camera_front,5,Needs investigation


In [8]:
missing_report = (
    df.isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})

)

missing_report = missing_report[
    missing_report["missing_count"] > 0
].copy()

missing_report["status"] = "Needs Investigation"

missing_report

,column,missing_count,status
3,avg_rating,101,Needs Investigation
5,processor_brand,20,Needs Investigation
6,num_cores,6,Needs Investigation
7,processor_speed,42,Needs Investigation
8,battery_capacity,11,Needs Investigation
10,fast_charging,211,Needs Investigation
16,os,14,Needs Investigation
18,primary_camera_front,5,Needs Investigation


In [89]:
from pathlib import Path

reports_dir = Path("../reports")
reports_dir.mkdir(parents = True, exist_ok = True)

suspicious_df.to_csv(
    reports_dir / "suspicious_values.csv",
    index = False

)

missing_report.to_csv(
    reports_dir / "missing_value_report.csv",
    index=False

)

In [9]:
print(suspicious_df)

     row_index      column  value                            reason  \
0           40  avg_rating    NaN  Outside the expected basic range   
1           43  avg_rating    NaN  Outside the expected basic range   
2           50  avg_rating    NaN  Outside the expected basic range   
3           55  avg_rating    NaN  Outside the expected basic range   
4           59  avg_rating    NaN  Outside the expected basic range   
..         ...         ...    ...                               ...   
96         857  avg_rating    NaN  Outside the expected basic range   
97         873  avg_rating    NaN  Outside the expected basic range   
98         874  avg_rating    NaN  Outside the expected basic range   
99         875  avg_rating    NaN  Outside the expected basic range   
100        906  avg_rating    NaN  Outside the expected basic range   

                  status  
0    Needs investigation  
1    Needs investigation  
2    Needs investigation  
3    Needs investigation  
4    Needs i

In [10]:
df["avg_rating"].describe()

count    879.000000
mean       7.825825
std        0.740285
min        6.000000
25%        7.400000
50%        8.000000
75%        8.400000
max        8.900000
Name: avg_rating, dtype: float64

In [11]:
df.isna().sum()

brand_name                     0
model                          0
price                          0
avg_rating                   101
5G_or_not                      0
processor_brand               20
num_cores                      6
processor_speed               42
battery_capacity              11
fast_charging_available        0
fast_charging                211
ram_capacity                   0
internal_memory                0
screen_size                    0
refresh_rate                   0
num_rear_cameras               0
os                            14
primary_camera_rear            0
primary_camera_front           5
extended_memory_available      0
resolution_height              0
resolution_width               0
dtype: int64